# So sánh trực tiếp ViT và Residual RVQ-VAE trên dữ liệu video

Notebook chỉ dùng dữ liệu ViTPose `250×60`. `direct_video_vit` nhận thẳng dữ liệu liên tục; `rvq_tfidf_grammar_mlp` dùng Residual RVQ-VAE để sinh token rồi tạo TF-IDF với unigram, bigram, trigram, transition và dwell-time trước khi phân loại bằng MLP.

In [ ]:
import os
from pathlib import Path

DEFAULT_DATA_ROOT = "/kaggle/input/datasets/xndung/fidgety/data"
DEFAULT_SOURCE_DIR = "/kaggle/working/Fidgety"
DEFAULT_OUTPUT_DIR = "/kaggle/working"
DATA_ROOT = Path(os.environ.get("FIDGETY_DATA_ROOT", DEFAULT_DATA_ROOT))
REPO_URL = "https://github.com/xn-dung/Fidgety.git"
SOURCE_DIR = Path(os.environ.get("FIDGETY_SOURCE_DIR", DEFAULT_SOURCE_DIR))
UPLOADED_PROJECT_DIR = None
RUN_MODE = os.environ.get("FIDGETY_RUN_MODE", "smoke")
SEED = 42
GPU_WORKERS = 2
CPU_THREADS_PER_WORKER = 2
OUTPUT_DIR = Path(os.environ.get("FIDGETY_OUTPUT_ROOT", DEFAULT_OUTPUT_DIR)) / f"direct_vit_vs_rvq_grammar_{RUN_MODE}_v7"

if RUN_MODE == "smoke":
    FOLDS = [1]
    VQVAE_EPOCHS = 8
    VIT_EPOCHS = 15
    VQVAE_PATIENCE = 4
    VIT_PATIENCE = 5
elif RUN_MODE == "full":
    FOLDS = list(range(1, 10))
    VQVAE_EPOCHS = 120
    VIT_EPOCHS = 120
    VQVAE_PATIENCE = 12
    VIT_PATIENCE = 15
else:
    raise ValueError("RUN_MODE must be smoke or full")

In [ ]:
import subprocess
import sys

if UPLOADED_PROJECT_DIR is not None:
    PROJECT_DIR = Path(UPLOADED_PROJECT_DIR)
else:
    if SOURCE_DIR.exists():
        subprocess.run(["git", "-C", str(SOURCE_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(SOURCE_DIR)], check=True)
    PROJECT_DIR = SOURCE_DIR / "đồ án"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(PROJECT_DIR))

import json
import math
import pickle
import random
import shutil
import threading
from concurrent.futures import ThreadPoolExecutor
from queue import Empty, Queue

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from data import TextStore, load_video_fold

if not DATA_ROOT.is_dir() and (DATA_ROOT.parent / "data.zip").is_file():
    DATA_ROOT = DATA_ROOT.parent / "data.zip"
if not DATA_ROOT.exists():
    raise FileNotFoundError(DATA_ROOT)
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Kaggle Settings")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VISIBLE_GPUS = torch.cuda.device_count()
WORKER_COUNT = min(GPU_WORKERS, VISIBLE_GPUS, len(FOLDS))
torch.set_num_threads(max(1, CPU_THREADS_PER_WORKER * WORKER_COUNT))
print("Data:", DATA_ROOT)
for gpu in range(VISIBLE_GPUS):
    print(f"GPU {gpu}: {torch.cuda.get_device_name(gpu)}")
print("Parallel workers:", WORKER_COUNT)

## Mô hình

In [ ]:
VERSION = "direct-vit-vs-residual-rvq-grammar-v7"
CODEBOOK_SIZE = 64
QUANTIZERS = 2
LATENT_DIM = 128
GRAMMAR_ORDER = 3
GRAMMAR_MAX_FEATURES = 40000
VQVAE_BATCH_SIZE = 32
VIT_BATCH_SIZE = 32

class ResidualBlock1D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv1d(channels, channels, 3, padding=1),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Conv1d(channels, channels, 3, padding=1),
            nn.BatchNorm1d(channels)
        )

    def forward(self, values):
        return F.gelu(values + self.layers(values))

class EMACodebook(nn.Module):
    def __init__(self, size, dimension, decay=0.99, epsilon=1e-5):
        super().__init__()
        self.size = size
        self.decay = decay
        self.epsilon = epsilon
        initial = torch.randn(size, dimension) / math.sqrt(dimension)
        self.register_buffer("embedding", initial)
        self.register_buffer("cluster_size", torch.zeros(size))
        self.register_buffer("embedding_sum", initial.clone())

    def forward(self, values):
        flat = values.reshape(-1, values.shape[-1])
        search_values = flat.to(self.embedding.dtype)
        with torch.autocast(device_type=values.device.type, enabled=False):
            distances = search_values.pow(2).sum(1, keepdim=True) + self.embedding.pow(2).sum(1) - 2 * search_values @ self.embedding.t()
        indices = distances.argmin(1)
        quantized = F.embedding(indices, self.embedding).reshape_as(values)
        if self.training:
            one_hot = F.one_hot(indices, self.size).to(self.embedding.dtype)
            counts = one_hot.sum(0)
            sums = one_hot.t() @ search_values.detach()
            self.cluster_size.mul_(self.decay).add_(counts, alpha=1 - self.decay)
            self.embedding_sum.mul_(self.decay).add_(sums, alpha=1 - self.decay)
            total = self.cluster_size.sum()
            smoothed = (self.cluster_size + self.epsilon) / (total + self.size * self.epsilon) * total
            self.embedding.copy_(self.embedding_sum / smoothed.unsqueeze(1).clamp_min(self.epsilon))
            dead = self.cluster_size < 0.1
            if dead.any() and len(search_values):
                samples = search_values[torch.randint(len(search_values), (int(dead.sum()),), device=search_values.device)]
                self.embedding[dead] = samples
                self.embedding_sum[dead] = samples
                self.cluster_size[dead] = 1.0
        commitment = F.mse_loss(values.float(), quantized.detach())
        return quantized.to(values.dtype), indices.reshape(values.shape[:-1]), commitment

class ResidualEMAQuantizer(nn.Module):
    def __init__(self, levels, size, dimension, commitment_weight=0.25):
        super().__init__()
        self.commitment_weight = commitment_weight
        self.codebooks = nn.ModuleList([EMACodebook(size, dimension) for _ in range(levels)])

    def forward(self, values):
        residual = values
        quantized_sum = torch.zeros_like(values)
        token_levels = []
        commitment = values.new_tensor(0.0)
        for codebook in self.codebooks:
            quantized, indices, level_loss = codebook(residual)
            quantized_sum = quantized_sum + quantized
            residual = residual - quantized.detach()
            token_levels.append(indices)
            commitment = commitment + level_loss
        straight_through = values + (quantized_sum - values).detach()
        return straight_through, self.commitment_weight * commitment, torch.stack(token_levels, dim=-1)

class ResidualRVQVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(60, 128, 5, padding=2),
            ResidualBlock1D(128),
            nn.Conv1d(128, 192, 4, stride=2, padding=1),
            ResidualBlock1D(192),
            nn.Conv1d(192, 256, 3, stride=2, padding=1),
            ResidualBlock1D(256),
            nn.Conv1d(256, LATENT_DIM, 3, padding=1)
        )
        self.quantizer = ResidualEMAQuantizer(QUANTIZERS, CODEBOOK_SIZE, LATENT_DIM)
        self.decoder = nn.Sequential(
            nn.Conv1d(LATENT_DIM, 256, 3, padding=1),
            ResidualBlock1D(256),
            nn.ConvTranspose1d(256, 192, 3, stride=2, padding=1),
            ResidualBlock1D(192),
            nn.ConvTranspose1d(192, 128, 4, stride=2, padding=1),
            ResidualBlock1D(128),
            nn.Conv1d(128, 60, 5, padding=2)
        )

    def encode(self, values):
        return self.encoder(values.transpose(1, 2)).transpose(1, 2)

    def forward(self, values):
        latent = self.encode(values)
        quantized, commitment, tokens = self.quantizer(latent)
        reconstruction = self.decoder(quantized.transpose(1, 2)).transpose(1, 2)
        return reconstruction, commitment, tokens

    @torch.no_grad()
    def tokens(self, values):
        return self.quantizer(self.encode(values))[2]

class DirectVideoViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.input_projection = nn.Linear(60, LATENT_DIM)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, LATENT_DIM))
        self.position = nn.Parameter(torch.zeros(1, 251, LATENT_DIM))
        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.position, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=LATENT_DIM, nhead=4, dim_feedforward=256, dropout=0.25, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=3)
        self.norm = nn.LayerNorm(LATENT_DIM)
        self.output = nn.Sequential(nn.Linear(LATENT_DIM, 64), nn.GELU(), nn.Dropout(0.35), nn.Linear(64, 1))

    def forward(self, values):
        embedded = self.input_projection(values)
        cls = self.cls_token.expand(len(values), -1, -1)
        encoded = self.encoder(torch.cat([cls, embedded], dim=1) + self.position)
        return self.output(self.norm(encoded[:, 0])).squeeze(1)

## Huấn luyện và grammar

In [ ]:
def seed_worker(seed, device):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.set_device(device)
    torch.cuda.manual_seed(seed)

def value_loader(values, labels=None, batch_size=32, shuffle=False, seed=0):
    tensors = [torch.as_tensor(values, dtype=torch.float32)]
    if labels is not None:
        tensors.append(torch.as_tensor(labels, dtype=torch.float32))
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(TensorDataset(*tensors), batch_size=batch_size, shuffle=shuffle, generator=generator, pin_memory=True)

def cpu_state(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

def classification_metrics(labels, probabilities, threshold):
    labels = np.asarray(labels, dtype=np.int64)
    predictions = (np.asarray(probabilities) >= threshold).astype(np.int64)
    tn = int(np.sum((labels == 0) & (predictions == 0)))
    fp = int(np.sum((labels == 0) & (predictions == 1)))
    fn = int(np.sum((labels == 1) & (predictions == 0)))
    tp = int(np.sum((labels == 1) & (predictions == 1)))
    sensitivity = tp / (tp + fn) if tp + fn else float("nan")
    specificity = tn / (tn + fp) if tn + fp else float("nan")
    return {"sensitivity": sensitivity, "specificity": specificity, "balanced_accuracy": (sensitivity + specificity) / 2, "tn": tn, "fp": fp, "fn": fn, "tp": tp}, predictions

def best_threshold(labels, probabilities):
    candidates = np.linspace(0.05, 0.95, 181)
    scores = np.asarray([classification_metrics(labels, probabilities, threshold)[0]["balanced_accuracy"] for threshold in candidates])
    best = np.flatnonzero(np.isclose(scores, scores.max()))
    index = int(best[np.argmin(np.abs(candidates[best] - 0.5))])
    return float(candidates[index]), float(scores[index])

def vqvae_loss(model, batch):
    reconstruction, commitment, _ = model(batch)
    position = F.mse_loss(reconstruction[:, :, :30], batch[:, :, :30])
    velocity = F.mse_loss(reconstruction[:, :, 30:], batch[:, :, 30:])
    return position + velocity + commitment

def train_vqvae(model, fit_values, val_values, device, seed):
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
    scaler = torch.cuda.amp.GradScaler()
    fit_loader = value_loader(fit_values, batch_size=VQVAE_BATCH_SIZE, shuffle=True, seed=seed)
    val_loader = value_loader(val_values, batch_size=VQVAE_BATCH_SIZE)
    best_loss = float("inf")
    best_state = cpu_state(model)
    stale = 0
    history = []
    for epoch in range(1, VQVAE_EPOCHS + 1):
        model.train()
        total = 0.0
        count = 0
        for (batch,) in fit_loader:
            batch = batch.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                loss = vqvae_loss(model, batch)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total += float(loss.detach()) * len(batch)
            count += len(batch)
        model.eval()
        val_total = 0.0
        val_count = 0
        with torch.no_grad():
            for (batch,) in val_loader:
                batch = batch.to(device, non_blocking=True)
                with torch.cuda.amp.autocast():
                    loss = vqvae_loss(model, batch)
                val_total += float(loss) * len(batch)
                val_count += len(batch)
        train_loss = total / count
        val_loss = val_total / val_count
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        if epoch == 1 or epoch % 10 == 0:
            print(f"RVQ-VAE epoch={epoch} train={train_loss:.5f} val={val_loss:.5f}", flush=True)
        if val_loss < best_loss - 1e-5:
            best_loss = val_loss
            best_state = cpu_state(model)
            stale = 0
        else:
            stale += 1
            if stale >= VQVAE_PATIENCE:
                break
    model.load_state_dict(best_state)
    return history, best_loss

@torch.no_grad()
def encode_tokens(model, values, device):
    model.eval()
    batches = []
    for (batch,) in value_loader(values, batch_size=128):
        batches.append(model.tokens(batch.to(device, non_blocking=True)).cpu().numpy())
    return np.concatenate(batches)

@torch.no_grad()
def vit_probabilities(model, values, device):
    model.eval()
    output = []
    for (batch,) in value_loader(values, batch_size=64):
        output.append(torch.sigmoid(model(batch.to(device, non_blocking=True))).cpu().numpy())
    return np.concatenate(output)

def train_direct_vit(model, fit_values, fit_labels, val_values, val_labels, test_values, test_labels, device, seed):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    negatives = float(np.sum(np.asarray(fit_labels) == 0))
    positives = float(np.sum(np.asarray(fit_labels) == 1))
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(negatives / positives, device=device))
    loader = value_loader(fit_values, fit_labels, VIT_BATCH_SIZE, True, seed)
    scaler = torch.cuda.amp.GradScaler()
    best_score = -1.0
    best_state = cpu_state(model)
    selected_threshold = 0.5
    selected_epoch = 0
    stale = 0
    for epoch in range(1, VIT_EPOCHS + 1):
        model.train()
        for values, labels in loader:
            values = values.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                loss = criterion(model(values), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        val_probabilities = vit_probabilities(model, val_values, device)
        threshold, score = best_threshold(val_labels, val_probabilities)
        if score > best_score + 1e-6:
            best_score = score
            best_state = cpu_state(model)
            selected_threshold = threshold
            selected_epoch = epoch
            stale = 0
        else:
            stale += 1
            if stale >= VIT_PATIENCE:
                break
    model.load_state_dict(best_state)
    probabilities = vit_probabilities(model, test_values, device)
    result, predictions = classification_metrics(test_labels, probabilities, selected_threshold)
    return model, result, predictions, probabilities, selected_threshold, selected_epoch, best_score

def dwell_bucket(length):
    if length == 1:
        return "1"
    if length == 2:
        return "2"
    if length <= 4:
        return "3-4"
    if length <= 8:
        return "5-8"
    return "9+"

def dwell_features(sequence, prefix):
    features = []
    start = 0
    previous = None
    while start < len(sequence):
        end = start + 1
        while end < len(sequence) and sequence[end] == sequence[start]:
            end += 1
        token = sequence[start]
        duration = dwell_bucket(end - start)
        features.append(f"{prefix}dw_{token}_{duration}")
        if previous is not None:
            features.append(f"{prefix}tr_{previous}_{token}")
            features.append(f"{prefix}trdw_{previous}_{token}_{duration}")
        previous = token
        start = end
    return features

def grammar_documents(tokens):
    documents = []
    for sample in tokens:
        words = []
        steps, levels = sample.shape
        for level in range(levels):
            sequence = [f"l{level}c{int(sample[step, level])}" for step in range(steps)]
            for order in range(1, GRAMMAR_ORDER + 1):
                words.extend(f"l{level}g{order}_" + "_".join(sequence[start:start + order]) for start in range(steps - order + 1))
            words.extend(dwell_features(sequence, f"l{level}"))
        joint = ["j" + "_".join(str(int(code)) for code in sample[step]) for step in range(steps)]
        for order in range(1, GRAMMAR_ORDER + 1):
            words.extend(f"jg{order}_" + "__".join(joint[start:start + order]) for start in range(steps - order + 1))
        words.extend(dwell_features(joint, "j"))
        documents.append(" ".join(words))
    return documents

def codebook_statistics(tokens):
    rows = []
    for level in range(tokens.shape[-1]):
        counts = np.bincount(tokens[:, :, level].reshape(-1), minlength=CODEBOOK_SIZE).astype(float)
        probabilities = counts[counts > 0] / counts.sum()
        rows.append({"level": level + 1, "used_codes": int(np.count_nonzero(counts)), "perplexity": float(np.exp(-np.sum(probabilities * np.log(probabilities))))})
    return rows

## Chạy 9-fold trên tối đa hai GPU

In [ ]:
initialization_lock = threading.Lock()

def train_fold(fold, gpu):
    fold_dir = OUTPUT_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    cache_path = fold_dir / "result.json"
    if cache_path.is_file():
        cached = json.loads(cache_path.read_text())
        if cached.get("version") == VERSION and len(cached.get("rows", [])) == 2:
            print(f"GPU {gpu} fold {fold} skipped", flush=True)
            return cached["rows"]
    device = torch.device(f"cuda:{gpu}")
    seed_worker(SEED + fold, device)
    data = load_video_fold(TextStore(DATA_ROOT), fold)
    rows = []
    with initialization_lock:
        torch.manual_seed(SEED + fold)
        vit = DirectVideoViT()
    vit, result, predictions, probabilities, threshold, epoch, val_score = train_direct_vit(vit, data.x_fit, data.y_fit, data.x_val, data.y_val, data.x_test, data.y_test, device, SEED + fold)
    rows.append({"fold": fold, "classifier": "direct_video_vit", "representation": "continuous_250x60", "train_samples": len(data.y_fit), "test_samples": len(data.y_test), **result})
    torch.save({"state_dict": cpu_state(vit), "threshold": threshold, "best_epoch": epoch, "validation_balanced_accuracy": val_score}, fold_dir / "direct_video_vit.pt")
    pd.DataFrame({"y_true": data.y_test, "probability": probabilities, "y_pred": predictions}).to_csv(fold_dir / "direct_video_vit_predictions.csv", index=False)
    print(f"GPU {gpu} fold={fold} direct_video_vit BA={result['balanced_accuracy']:.4f}", flush=True)
    del vit
    torch.cuda.empty_cache()
    with initialization_lock:
        torch.manual_seed(SEED + fold + 100)
        rvqvae = ResidualRVQVAE()
    rvqvae.to(device)
    history, best_reconstruction = train_vqvae(rvqvae, data.x_fit, data.x_val, device, SEED + fold + 100)
    train_tokens = encode_tokens(rvqvae, data.x_train, device)
    test_tokens = encode_tokens(rvqvae, data.x_test, device)
    split = len(data.y_fit)
    fit_tokens = train_tokens[:split]
    val_tokens = train_tokens[split:]
    fit_documents = grammar_documents(fit_tokens)
    val_documents = grammar_documents(val_tokens)
    test_documents = grammar_documents(test_tokens)
    vectorizer = TfidfVectorizer(lowercase=False, token_pattern=r"[^ ]+", sublinear_tf=True, norm="l2", min_df=2, max_features=GRAMMAR_MAX_FEATURES, dtype=np.float32)
    fit_matrix = vectorizer.fit_transform(fit_documents)
    val_matrix = vectorizer.transform(val_documents)
    test_matrix = vectorizer.transform(test_documents)
    classifier = MLPClassifier(hidden_layer_sizes=(256, 64), max_iter=1000, early_stopping=True, validation_fraction=0.125, n_iter_no_change=25, random_state=SEED + fold)
    classifier.fit(fit_matrix, data.y_fit)
    val_probabilities = classifier.predict_proba(val_matrix)[:, 1]
    threshold, val_score = best_threshold(data.y_val, val_probabilities)
    probabilities = classifier.predict_proba(test_matrix)[:, 1]
    result, predictions = classification_metrics(data.y_test, probabilities, threshold)
    rows.append({"fold": fold, "classifier": "rvq_tfidf_grammar_mlp", "representation": f"residual_rvq_{QUANTIZERS}x{CODEBOOK_SIZE}", "train_samples": len(data.y_fit), "test_samples": len(data.y_test), "vocabulary_size": len(vectorizer.vocabulary_), **result})
    torch.save({"version": VERSION, "state_dict": cpu_state(rvqvae), "best_validation_loss": best_reconstruction}, fold_dir / "residual_rvqvae.pt")
    pd.DataFrame(history).to_csv(fold_dir / "rvqvae_history.csv", index=False)
    (fold_dir / "codebook_statistics.json").write_text(json.dumps(codebook_statistics(train_tokens), indent=2))
    with (fold_dir / "rvq_tfidf_grammar_mlp.pkl").open("wb") as stream:
        pickle.dump({"vectorizer": vectorizer, "classifier": classifier, "threshold": threshold, "grammar_order": GRAMMAR_ORDER}, stream)
    pd.DataFrame({"y_true": data.y_test, "probability": probabilities, "y_pred": predictions}).to_csv(fold_dir / "rvq_tfidf_grammar_mlp_predictions.csv", index=False)
    print(f"GPU {gpu} fold={fold} rvq_tfidf_grammar_mlp BA={result['balanced_accuracy']:.4f}", flush=True)
    cache_path.write_text(json.dumps({"version": VERSION, "rows": rows}, indent=2))
    del rvqvae
    torch.cuda.empty_cache()
    return rows

fold_queue = Queue()
for fold in FOLDS:
    fold_queue.put(fold)

def gpu_worker(gpu):
    rows = []
    while True:
        try:
            fold = fold_queue.get_nowait()
        except Empty:
            return rows
        print(f"GPU {gpu} started fold {fold}", flush=True)
        rows.extend(train_fold(fold, gpu))
        fold_queue.task_done()

with ThreadPoolExecutor(max_workers=WORKER_COUNT) as executor:
    futures = [executor.submit(gpu_worker, gpu) for gpu in range(WORKER_COUNT)]
    result_rows = [row for future in futures for row in future.result()]

fold_results = pd.DataFrame(result_rows).sort_values(["classifier", "fold"])
fold_results.to_csv(OUTPUT_DIR / "fold_results.csv", index=False)

## Bảng so sánh

In [ ]:
def summarize(frame):
    rows = []
    for classifier, group in frame.groupby("classifier"):
        row = {"classifier": classifier, "representation": group["representation"].iloc[0], "n_folds": len(group)}
        for metric_name in ("sensitivity", "specificity", "balanced_accuracy"):
            values = group[metric_name].to_numpy(float)
            mean = float(values.mean())
            margin = float(1.96 * values.std(ddof=1) / np.sqrt(len(values))) if len(values) > 1 else float("nan")
            row[metric_name + "_mean"] = mean
            row[metric_name + "_ci_low"] = mean - margin
            row[metric_name + "_ci_high"] = mean + margin
        row["delta_vs_retrained"] = row["balanced_accuracy_mean"] - 0.89402
        rows.append(row)
    return pd.DataFrame(rows)

summary = summarize(fold_results)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
comparison = summary[["classifier", "representation", "n_folds", "balanced_accuracy_mean", "balanced_accuracy_ci_low", "balanced_accuracy_ci_high", "delta_vs_retrained"]].copy()
references = pd.DataFrame([
    {"classifier": "author_cnn_retrained", "representation": "continuous_250x60", "n_folds": 9, "balanced_accuracy_mean": 0.89402, "balanced_accuracy_ci_low": np.nan, "balanced_accuracy_ci_high": np.nan, "delta_vs_retrained": 0.0},
    {"classifier": "author_cnn_published", "representation": "continuous_250x60", "n_folds": 9, "balanced_accuracy_mean": 0.9066064667, "balanced_accuracy_ci_low": np.nan, "balanced_accuracy_ci_high": np.nan, "delta_vs_retrained": 0.9066064667 - 0.89402}
])
comparison = pd.concat([comparison, references], ignore_index=True)
comparison["balanced_accuracy_mean_pct"] = 100 * comparison["balanced_accuracy_mean"]
comparison["delta_vs_retrained_pp"] = 100 * comparison["delta_vs_retrained"]
comparison = comparison.sort_values("balanced_accuracy_mean", ascending=False)
comparison.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
display(fold_results.round(4))
display(summary.round(4))
display(comparison[["classifier", "representation", "n_folds", "balanced_accuracy_mean_pct", "delta_vs_retrained_pp"]].round(3))

positions = np.arange(len(summary))
values = 100 * summary["balanced_accuracy_mean"].to_numpy()
errors = 100 * (summary["balanced_accuracy_ci_high"] - summary["balanced_accuracy_mean"]).to_numpy()
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(positions, values, yerr=errors if np.isfinite(errors).all() else None, capsize=5)
ax.axhline(89.402, color="tab:red", linestyle=":", label="Author CNN retrained: 89.40%")
ax.axhline(90.6606467, color="black", linestyle="--", label="Author CNN published: 90.66%")
ax.set_xticks(positions, summary["classifier"], rotation=15, ha="right")
ax.set_ylabel("Balanced accuracy (%)")
ax.set_ylim(0, 100)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "comparison.png", dpi=180)
plt.show()
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("Artifacts:", archive)